# Disaster role adapter — Gemma 4 QLoRA
Training only. This notebook does not deploy a service or call Vertex AI. Use synthetic/licensed data.


In [ ]:
%pip install -q torch tensorboard torchao "transformers>=5.10.1" datasets accelerate evaluate bitsandbytes trl "peft>=0.19.0" protobuf sentencepiece


In [ ]:
from pathlib import Path
import json, torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

ROLE = 'resident'
BASE_MODEL = 'google/gemma-4-12B'
PROCESSOR_MODEL = 'google/gemma-4-12B-it'
TRAIN_JSONL = '/content/train.jsonl'
OUTPUT_DIR = f'/content/{ROLE}-adapter'
DATASET_LABEL = 'synthetic-v1'
assert torch.cuda.is_available(), 'GPU runtime required'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), 'GiB')


In [ ]:
dataset = load_dataset('json', data_files=TRAIN_JSONL, split='train')
if len(dataset) < 5:
    raise ValueError('Need more than a tiny smoke fixture before real training')
dataset = dataset.train_test_split(test_size=min(0.2, max(1/len(dataset), 0.05)), seed=42)
print(dataset)


In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=dtype, bnb_4bit_quant_storage=dtype)
model = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL, device_map='auto', dtype=dtype, quantization_config=quant)
processor = AutoProcessor.from_pretrained(PROCESSOR_MODEL)
if torch.cuda.get_device_properties(0).total_memory / 1024**3 > 16:
    model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(lora_alpha=16, lora_dropout=0.05, r=16, bias='none', task_type='CAUSAL_LM', modules_to_save=['lm_head','embed_tokens'], ensure_weight_tying=True)

def collate_fn(examples):
    texts = [processor.apply_chat_template(x['messages'], add_generation_prompt=False, tokenize=False).strip() for x in examples]
    batch = processor(text=texts, return_tensors='pt', padding=True)
    labels = batch['input_ids'].clone()
    marker = [processor.tokenizer.convert_tokens_to_ids('<|turn>'), processor.tokenizer.convert_tokens_to_ids('model'), processor.tokenizer.convert_tokens_to_ids('\n')]
    for i, row in enumerate(batch['input_ids'].tolist()):
        start = None
        for j in range(len(row) - len(marker) + 1):
            if row[j:j+len(marker)] == marker:
                start = j + len(marker); break
        if start is not None:
            labels[i, :start] = -100
        labels[i, labels[i] == processor.tokenizer.pad_token_id] = -100
    batch['labels'] = labels
    return batch

args = SFTConfig(output_dir=OUTPUT_DIR, max_length=1024, num_train_epochs=2, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=2e-4, logging_steps=10, save_strategy='epoch', eval_strategy='epoch', fp16=dtype==torch.float16, bf16=dtype==torch.bfloat16, lr_scheduler_type='constant', push_to_hub=False, report_to='tensorboard', dataset_kwargs={'skip_prepare_dataset': True}, remove_unused_columns=False)
trainer = SFTTrainer(model=model, args=args, train_dataset=dataset['train'], eval_dataset=dataset['test'], peft_config=peft_config, processing_class=processor, data_collator=collate_fn)


In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
manifest = {'schema_version':'role-adapter-manifest/v1','role':ROLE,'base_model':BASE_MODEL,'processor_model':PROCESSOR_MODEL,'dataset_label':DATASET_LABEL,'method':'qlora-4bit','lora_r':16,'lora_alpha':16,'epochs':2,'max_length':1024}
Path(OUTPUT_DIR, 'adapter_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('saved', OUTPUT_DIR)
